### **Sprint - Engenharia de Dados**

Nome: Aline Fiori Gonçalves

Matrícula: 4052025000106

Dataset: Pregnancy

# 1. Contexto de Negócio

O presente projeto tem como objetivo construir um pipeline de dados estruturado no ambiente Databricks para apoiar a análise de indicadores de risco de saúde materna. O ecossistema de dados visa coletar, validar, transformar e catalogar atributos clínicos para responder a questões cruciais sobre grupos de risco gestacional.

### Origem e Estrutura dos Dados Brutos
Os dados originais consistem em medições de saúde física coletadas em gestantes (como pressão arterial, níveis de glicose, temperatura corporal e frequência cardíaca) e o nível de risco associado (`RiskLevel`). Os dados brutos são ingeridos originalmente em formato CSV.

Para fundamentar o contexto de negócio e mapear os atributos clínicos recebidos antes das etapas formais de ETL, realizou-se a inspeção técnica da fonte de dados bruta no Databricks via PySpark.

### Licença dos Dados e Termos de Uso (Governança e Conformidade)
O conjunto de dados utilizado neste projeto está sob a licença **Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)**.

Essa licença estabelece as seguintes condições de uso:
* **Atribuição (BY):** É necessário dar o devido crédito à fonte original dos dados.
* **NãoComercial (NC):** O uso deste *dataset* é estritamente limitado a fins acadêmicos, de estudo e pesquisa, sendo proibido qualquer tipo de exploração comercial.
* **CompartilhaIgual (SA):** Caso os dados sejam alterados, transformados ou servirem de base para novas criações, o trabalho derivado deve ser distribuído sob a mesma licença da obra original.

Conforme as diretrizes da licença, este projeto é desenvolvido exclusivamente para fins didáticos e acadêmicos dentro do escopo da Sprint de Engenharia de Dados.

In [0]:
# ==============================================================================
# IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
import urllib.request
import os
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, lower, trim, when
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 1. Definição explícita do Schema
maternal_schema = StructType([
    StructField("Age", IntegerType(), True),
    StructField("SystolicBP", IntegerType(), True),
    StructField("DiastolicBP", IntegerType(), True),
    StructField("BS", DoubleType(), True),
    StructField("BodyTemp", DoubleType(), True),
    StructField("HeartRate", IntegerType(), True),
    StructField("RiskLevel", StringType(), True)
])

# 2. URL do arquivo CSV "Raw" no GitHub (URL contínua sem quebras de linha)
url_github = "https://raw.githubusercontent.com/AlineFiori/MVP---Engenharia-de-Dados/main/Maternal%20Health%20Risk%20Data%20Set.csv"

# 3. Define o caminho relativo dentro do seu próprio diretório no Workspace
# O os.getcwd() identifica automaticamente a pasta atual onde o notebook está rodando
dir_atual = os.getcwd()
path_local_csv = os.path.join(dir_atual, "Maternal_Health_Risk_Data_Set.csv")

# Download via Python diretamente para o diretório do Workspace
urllib.request.urlretrieve(url_github, path_local_csv)

# 4. Leitura no Spark utilizando o caminho relativo do Workspace
df_raw = spark.read \
    .option("header", "true") \
    .schema(maternal_schema) \
    .csv(f"file:{path_local_csv}")

# Exibição do Schema e dos dados
df_raw.printSchema()
display(df_raw)

root
 |-- Age: integer (nullable = true)
 |-- SystolicBP: integer (nullable = true)
 |-- DiastolicBP: integer (nullable = true)
 |-- BS: double (nullable = true)
 |-- BodyTemp: double (nullable = true)
 |-- HeartRate: integer (nullable = true)
 |-- RiskLevel: string (nullable = true)



Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
25,130,80,15.0,98.0,86,high risk
35,140,90,13.0,98.0,70,high risk
29,90,70,8.0,100.0,80,high risk
30,140,85,7.0,98.0,70,high risk
35,120,60,6.1,98.0,76,low risk
23,140,80,7.01,98.0,70,high risk
23,130,70,7.01,98.0,78,mid risk
35,85,60,11.0,102.0,86,high risk
32,120,90,6.9,98.0,70,mid risk
42,130,80,18.0,98.0,70,high risk


# 2. Pipeline de Dados

O pipeline de dados foi construído seguindo a arquitetura em camadas no Databricks. Os dados são limpos, transformados, enriquecidos com metadados de ingestão e persistidos no formato otimizado **Delta Lake**.

---

### Arquitetura Medalhão (Medallion Architecture)

A Arquitetura Medalhão é um padrão de design de dados usado para organizar e estruturar os dados no *data lake*, melhorando progressivamente a qualidade e a estrutura das informações à medida que elas fluem pelas camadas. O objetivo principal é transformar dados brutos em ativos de alta qualidade, confiáveis e prontos para análises de negócios.

#### As Três Camadas do Pipeline:

1. **Camada Bronze (Raw / Bruta):**
   * **Conceito:** É o ponto de entrada dos dados. Armazena as informações exatamente como vêm das fontes originais (arquivos CSV armazenados na Workspace/GitHub), sem alterações estruturais.
   * **Objetivo no Projeto:** Garantir uma cópia de segurança e histórico imutável dos dados originais (`df_raw`) para auditoria ou reprocessamento.

2. **Camada Silver (Refined / Refinada):**
   * **Conceito:** Os dados da camada Bronze são limpos, filtrados, padronizados, deduplicados e enriquecidos.
   * **Objetivo no Projeto:** Executar as tratativas de qualidade (padronização de nomes em *snake_case*, tratamento da coluna `risk_level` com `trim()` e `lower()`, remoção de duplicatas/nulos e adição de `ingestion_timestamp`), salvando os dados tratados na tabela Delta **`stg_maternal_health_risk`**.

3. **Camada Gold (Curated / Consolidada):**
   * **Conceito:** A camada final, onde os dados são agregados e moldados para responder a perguntas de negócio específicas.
   * **Objetivo no Projeto:** Fornecer as visualizações e consultas SQL (`v_maternal_health`) que respondem aos indicadores de risco gestacional e faixas etárias maternas.

#### Principais Benefícios da Abordagem:
* **Qualidade Progressiva:** Aumento gradual da confiabilidade dos dados de ponta a ponta.
* **Rastreabilidade (Lineage):** Facilidade para auditar a origem e as transformações aplicadas em cada dado.
* **Flexibilidade e Reprocessamento:** Permite reprocessar as camadas Silver e Gold a partir dos dados brutos da camada Bronze sem a necessidade de reimportar da fonte externa.

## Carga dos Dados — Ingestão Bronze

In [0]:
# Célula da Camada Bronze (Ajustada)

# 1. Adiciona metadado de rastreabilidade utilizando o df_raw já carregado com schema explícito
df_bronze = df_raw.withColumn("data_ingestao", current_timestamp())

# 2. Salva no Catálogo/Metastore como Tabela Delta
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_pregnancy")

print("✅ Camada Bronze criada e persistida com sucesso!")
display(spark.table("bronze_pregnancy"))

✅ Camada Bronze criada e persistida com sucesso!


Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel,data_ingestao
25,130,80,15.0,98.0,86,high risk,2026-09-27T21:32:15.681Z
35,140,90,13.0,98.0,70,high risk,2026-09-27T21:32:15.681Z
29,90,70,8.0,100.0,80,high risk,2026-09-27T21:32:15.681Z
30,140,85,7.0,98.0,70,high risk,2026-09-27T21:32:15.681Z
35,120,60,6.1,98.0,76,low risk,2026-09-27T21:32:15.681Z
23,140,80,7.01,98.0,70,high risk,2026-09-27T21:32:15.681Z
23,130,70,7.01,98.0,78,mid risk,2026-09-27T21:32:15.681Z
35,85,60,11.0,102.0,86,high risk,2026-09-27T21:32:15.681Z
32,120,90,6.9,98.0,70,mid risk,2026-09-27T21:32:15.681Z
42,130,80,18.0,98.0,70,high risk,2026-09-27T21:32:15.681Z


## Qualidade de Dados & Camada Silver

In [0]:
# ==============================================================================
# CAMADA SILVER: Qualidade de Dados, Limpeza e Circuit Breaker
# ==============================================================================

# 1. Leitura da Camada Bronze
df_silver_raw = spark.read.table("bronze_pregnancy")

# 2. Normalização dos nomes de colunas (snake_case) e padronização de texto
df_silver_transformed = df_silver_raw \
    .withColumnRenamed("Age", "age") \
    .withColumnRenamed("SystolicBP", "systolic_bp") \
    .withColumnRenamed("DiastolicBP", "diastolic_bp") \
    .withColumnRenamed("BS", "blood_sugar") \
    .withColumnRenamed("BodyTemp", "body_temp_f") \
    .withColumnRenamed("HeartRate", "heart_rate") \
    .withColumn("risk_level", trim(lower(col("RiskLevel")))) \
    .drop("RiskLevel")

# 3. TESTE DE QUALIDADE CLINICA (Circuit Breaker)
# Valida limites fisiológicos e garante que a pressão sistólica seja maior que a diastólica
registros_invalidos = df_silver_transformed.filter(
    (col("age") < 10) | (col("age") > 100) | 
    (col("systolic_bp") <= 0) | (col("diastolic_bp") <= 0) |
    (col("systolic_bp") <= col("diastolic_bp"))
).count()

if registros_invalidos > 0:
    raise ValueError(f"🚨 ERRO DE QUALIDADE: Foram encontrados {registros_invalidos} registros clinicamente inválidos. Pipeline interrompido.")
else:
    print("✅ Testes de qualidade clínica aprovados com sucesso!")

# 4. Higienização, Deduplicação e Rastreabilidade
df_silver_clean = df_silver_transformed \
    .dropDuplicates() \
    .dropna(subset=["age", "risk_level"]) \
    .withColumn("ingestion_timestamp", current_timestamp())

# 5. Persistência e Catalogação na Camada Silver (Delta Lake)
df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_pregnancy")

print("✅ Camada Silver ('silver_pregnancy') criada e persistida no Delta Lake!")
display(spark.table("silver_pregnancy"))

✅ Testes de qualidade clínica aprovados com sucesso!
✅ Camada Silver ('silver_pregnancy') criada e persistida no Delta Lake!


age,systolic_bp,diastolic_bp,blood_sugar,body_temp_f,heart_rate,data_ingestao,risk_level,ingestion_timestamp
30,140,85,7.0,98.0,70,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z
23,140,80,7.01,98.0,70,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z
23,90,60,7.01,98.0,76,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z
19,120,75,7.2,98.0,66,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z
21,75,50,6.1,98.0,70,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z
26,85,60,6.0,101.0,86,2026-09-27T21:32:15.681Z,mid risk,2026-09-27T21:32:21.568Z
23,120,90,7.8,98.0,60,2026-09-27T21:32:15.681Z,mid risk,2026-09-27T21:32:21.568Z
50,140,95,17.0,98.0,60,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z
15,80,60,6.7,98.0,80,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z
40,120,90,12.0,98.0,80,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z


## Pipeline de Dados — Camada Gold

In [0]:


# 1. Leitura da Camada Silver
df_gold_raw = spark.read.table("silver_pregnancy")

# 2. Engenharia de Features
df_gold = df_gold_raw \
    .withColumn(
        "faixa_etaria",
        when(col("age") < 20, "Adolescente")
        .when((col("age") >= 20) & (col("age") < 35), "Adulta")
        .otherwise("Idade Materna Avançada")
    ) \
    .withColumn(
        "flag_hipertensao",
        when((col("systolic_bp") >= 140) | (col("diastolic_bp") >= 90), 1).otherwise(0)
    ) \
    .withColumn(
        "target_risk_encoded",
        when(col("risk_level") == "low risk", 0)
        .when(col("risk_level") == "mid risk", 1)
        .when(col("risk_level") == "high risk", 2)
        .otherwise(-1)
    )

# 3. Salva na Camada Gold (Delta Lake)
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_pregnancy_features")

print("✅ Camada Gold pronta para análise e BI!")
display(spark.table("gold_pregnancy_features"))

✅ Camada Gold pronta para análise e BI!


age,systolic_bp,diastolic_bp,blood_sugar,body_temp_f,heart_rate,data_ingestao,risk_level,ingestion_timestamp,faixa_etaria,flag_hipertensao,target_risk_encoded
30,140,85,7.0,98.0,70,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z,Adulta,1,2
23,140,80,7.01,98.0,70,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z,Adulta,1,2
23,90,60,7.01,98.0,76,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z,Adulta,0,0
19,120,75,7.2,98.0,66,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z,Adolescente,0,0
21,75,50,6.1,98.0,70,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z,Adulta,0,0
26,85,60,6.0,101.0,86,2026-09-27T21:32:15.681Z,mid risk,2026-09-27T21:32:21.568Z,Adulta,0,1
23,120,90,7.8,98.0,60,2026-09-27T21:32:15.681Z,mid risk,2026-09-27T21:32:21.568Z,Adulta,1,1
50,140,95,17.0,98.0,60,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z,Idade Materna Avançada,1,2
15,80,60,6.7,98.0,80,2026-09-27T21:32:15.681Z,low risk,2026-09-27T21:32:21.568Z,Adolescente,0,0
40,120,90,12.0,98.0,80,2026-09-27T21:32:15.681Z,high risk,2026-09-27T21:32:21.568Z,Idade Materna Avançada,1,2


## Tópico de Otimização e Manutenção

In [0]:
# Otimização dos arquivos no Delta Lake
tabelas = ["bronze_pregnancy", "silver_pregnancy", "gold_pregnancy_features"]

for tabela in tabelas:
    spark.sql(f"OPTIMIZE {tabela}")
    print(f"Otimização concluída na tabela: {tabela}")

print("✅ Manutenção e otimização do ecossistema concluídas!")

Otimização concluída na tabela: bronze_pregnancy
Otimização concluída na tabela: silver_pregnancy
Otimização concluída na tabela: gold_pregnancy_features
✅ Manutenção e otimização do ecossistema concluídas!


# 3. Análise de Dados
Nesta etapa, o conjunto de dados higienizado e persistido no Delta Lake é consultado via PySpark SQL para responder diretamente às 3 perguntas de negócio definidas no planejamento do projeto.

In [0]:
# ==============================================================================
# SEÇÃO 7: ANÁLISE DE DADOS (Consultas SQL)
# ==============================================================================

# Ajuste: Apontar diretamente para a camada Silver tratada no Delta Lake
spark.sql("SELECT * FROM silver_pregnancy").createOrReplaceTempView("v_maternal_health")

print("View 'v_maternal_health' criada com sucesso a partir da camada Silver!")

View 'v_maternal_health' criada com sucesso a partir da camada Silver!


Pergunta 1: Qual o percentual de gestantes com idade igual ou superior a 35 anos (35+)?

In [0]:
query_q1 = """
SELECT 
    COUNT(*) as total_pacientes,
    SUM(CASE WHEN age >= 35 THEN 1 ELSE 0 END) as pacientes_35_mais,
    ROUND((SUM(CASE WHEN age >= 35 THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) as percentual_35_mais
FROM v_maternal_health
"""
df_q1 = spark.sql(query_q1)
df_q1.show()

+---------------+-----------------+------------------+
|total_pacientes|pacientes_35_mais|percentual_35_mais|
+---------------+-----------------+------------------+
|            452|              137|             30.31|
+---------------+-----------------+------------------+



Pergunta 2: Qual a distribuição e o percentual de pacientes classificadas em alto risco (high risk)?

In [0]:
query_q2 = """
SELECT 
    risk_level,
    COUNT(*) as quantidade,
    ROUND((COUNT(*) * 100.0 / (SELECT COUNT(*) FROM v_maternal_health)), 2) as percentual
FROM v_maternal_health
GROUP BY risk_level
ORDER BY quantidade DESC
"""
df_q2 = spark.sql(query_q2)
df_q2.show()

+----------+----------+----------+
|risk_level|quantidade|percentual|
+----------+----------+----------+
|  low risk|       234|     51.77|
| high risk|       112|     24.78|
|  mid risk|       106|     23.45|
+----------+----------+----------+



Pergunta 3: Qual a relação entre a faixa etária materna e a prevalência de alto risco gestacional?

In [0]:
query_q3 = """
SELECT 
    CASE 
        WHEN age < 25 THEN '1. Menor que 25 anos'
        WHEN age BETWEEN 25 AND 34 THEN '2. 25 a 34 anos'
        ELSE '3. 35 anos ou mais'
    END as faixa_etaria,
    COUNT(*) as total_pacientes,
    SUM(CASE WHEN risk_level = 'high risk' THEN 1 ELSE 0 END) as qtd_alto_risco,
    ROUND((SUM(CASE WHEN risk_level = 'high risk' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) as pct_alto_risco
FROM v_maternal_health
GROUP BY 1
ORDER BY 1
"""
df_q3 = spark.sql(query_q3)
df_q3.show()

+--------------------+---------------+--------------+--------------+
|        faixa_etaria|total_pacientes|qtd_alto_risco|pct_alto_risco|
+--------------------+---------------+--------------+--------------+
|1. Menor que 25 anos|            219|            27|         12.33|
|     2. 25 a 34 anos|             96|            31|         32.29|
|  3. 35 anos ou mais|            137|            54|         39.42|
+--------------------+---------------+--------------+--------------+



# 4. Resposta às Hipóteses do Problema

In [0]:
# ==============================================================================
# ANÁLISE EXPLORATÓRIA & VALIDAÇÃO DAS HIPÓTESES
# ==============================================================================

df_spark = spark.table("silver_pregnancy")

# Mapeamento dinâmico do nome das colunas (funciona para 'Age' ou 'age', 'RiskLevel' ou 'risk_level')
col_age = "age" if "age" in df_spark.columns else "Age"
col_risk = "risk_level" if "risk_level" in df_spark.columns else "RiskLevel"

total_registros = df_spark.count()

# Hipótese 1: Percentual do grupo de risco de idade 35+
gestantes_35_mais = df_spark.filter(F.col(col_age) >= 35).count()
pct_35_mais = (gestantes_35_mais / total_registros) * 100

# Hipótese 2: Percentual de gestação de alto risco ('high risk') no dataset
alto_risco_total = df_spark.filter(F.lower(F.col(col_risk)) == "high risk").count()
pct_alto_risco = (alto_risco_total / total_registros) * 100

# Percentual de alto risco ESPECÍFICO no grupo 35+
alto_risco_35_mais = df_spark.filter(
    (F.col(col_age) >= 35) & (F.lower(F.col(col_risk)) == "high risk")
).count()

pct_alto_risco_no_grupo35 = (
    (alto_risco_35_mais / gestantes_35_mais) * 100 if gestantes_35_mais > 0 else 0
)

print(f"--- RESPOSTAS ÀS HIPÓTESES ---")
print(f"Total de registros analisados: {total_registros}")
print(f"1. Percentual de gestantes com 35+ anos: {pct_35_mais:.2f}% ({gestantes_35_mais} pacientes)")
print(f"2. Percentual geral de gestações de Alto Risco: {pct_alto_risco:.2f}%")
print(f"   -> Dentro do grupo 35+, a taxa de Alto Risco é de: {pct_alto_risco_no_grupo35:.2f}%")

# Hipótese 3: Mapeamento categórico para calcular a correlação entre Idade e Risco
df_corr = df_spark.withColumn(
    "risk_level_num",
    F.when(F.lower(F.col(col_risk)) == "low risk", 1)
     .when(F.lower(F.col(col_risk)) == "mid risk", 2)
     .when(F.lower(F.col(col_risk)) == "high risk", 3)
     .otherwise(0)
)

correlacao_idade_risco = df_corr.stat.corr(col_age, "risk_level_num")
print(f"3. Correlação de Pearson entre Idade Materna e Risco Gestacional: {correlacao_idade_risco:.4f}")

--- RESPOSTAS ÀS HIPÓTESES ---
Total de registros analisados: 452
1. Percentual de gestantes com 35+ anos: 30.31% (137 pacientes)
2. Percentual geral de gestações de Alto Risco: 24.78%
   -> Dentro do grupo 35+, a taxa de Alto Risco é de: 39.42%
3. Correlação de Pearson entre Idade Materna e Risco Gestacional: 0.1830


# 5. Modelagem Preditiva

In [0]:
# ==============================================================================
# SEÇÃO 9: PREPARAÇÃO DOS DADOS PARA MACHINE LEARNING
# ==============================================================================

# 1. Carregamento direto da Tabela Gold
try:
    df_spark_ml = spark.read.table("gold_pregnancy_features")
    print("-> Dados carregados com sucesso da camada Gold ('gold_pregnancy_features').")
except Exception as e:
    raise RuntimeError("Tabela 'gold_pregnancy_features' não encontrada no metastore.") from e

# 2. Conversão para Pandas
df_ml = df_spark_ml.toPandas()

# 3. Seleção exata das Features padronizadas e Variável Alvo
features = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar', 'body_temp_f', 'heart_rate', 'flag_hipertensao']

X = df_ml[features]
y = df_ml['target_risk_encoded']

print("✅ Features e Target carregados diretamente da camada Gold!")
print(f"Formato da matriz de entrada X: {X.shape}")

-> Dados carregados com sucesso da camada Gold ('gold_pregnancy_features').
✅ Features e Target carregados diretamente da camada Gold!
Formato da matriz de entrada X: (452, 7)



# 6. Avaliação

### Atingimento de Objetivos
O objetivo principal deste projeto — estruturar um pipeline end-to-end de Engenharia e Ciência de Dados no Databricks para análise de risco gestacional — foi totalmente atingido com sucesso[cite: 4]. 

Os dados foram ingeridos com validação de schema, limpos e organizados sob a **Arquitetura Medallion (Bronze, Silver e Gold)** no Delta Lake[cite: 4]. Essa estrutura viabilizou tanto consultas SQL analíticas de alta performance quanto a extração de *features* limpas para o treinamento e avaliação de modelos preditivos de Machine Learning, respondendo de forma robusta às hipóteses de negócio estabelecidas[cite: 4].

---

### Dificuldades Encontradas
Durante o ciclo de desenvolvimento, as principais barreiras técnicas superadas foram[cite: 4]:
* **Gerenciamento do Ambiente Databricks:** Configuração de cluster PySpark, tratamento de cotas e ajuste de tempo de execução para garantir escritas atômicas e seguras nas tabelas Delta Lake sem interrupções[cite: 4].
* **Padronização de Dados Não Estruturados/Texto:** Tratamento de inconsistências na coluna de risco (*RiskLevel*), resolvendo variações de maiúsculas/minúsculas e espaçamentos com funções PySpark antes da persistência na camada Silver[cite: 4].
* **Ajuste de Tipagem e Schemas:** Correção manual e coerção de tipos de dados numéricos (como inteiros e pontos flutuantes) para evitar perdas de precisão ou incompatibilidades nas agregações SQL e na vetorização de Machine Learning[cite: 4].

---

### Trabalhos Futuros
Para expandir o valor técnico da solução construída, propõem-se as seguintes melhorias evolutivas[cite: 4]:
* **Evolução do Pipeline para Delta Live Tables (DLT):** Refatorar o pipeline Medallion existente para o framework DLT, utilizando *Expectations* nativas para substituir e aprimorar as regras automatizadas do Circuit Breaker[cite: 4].
* **Automação de CI/CD e Governança:** Implementar esteiras de implantação contínua utilizando *Databricks Asset Bundles (DABs)* e expandir o controle de linhagem de dados via *Unity Catalog*[cite: 4].
* **Dashboards e Monitoramento de Drift:** Criar painéis no Databricks SQL para acompanhamento diário das gestantes e implementar monitoramento automatizado de *data drift* e *concept drift* no MLflow para os modelos em produção[cite: 4].

---

# 7. Conclusão e Considerações Finais

O desenvolvimento deste trabalho demonstrou a construção completa de uma solução de dados moderna no Databricks, combinando rigor técnico de Engenharia de Dados com o valor analítico e preditivo da Ciência de Dados aplicada à saúde materno-infantil[cite: 4].

---

### Síntese dos Resultados e Entregas

#### 1. Engenharia de Dados & Arquitetura Medallion
* **Camada Bronze (Raw):** Garantiu a ingestão imutável e auditável dos dados brutos em formato Delta Lake, incluindo metadados temporais (`data_ingestao`) para rastreabilidade[cite: 4].
* **Camada Silver (Clean & Trusted):** Implementou uma barreira de qualidade automatizada (**Circuit Breaker**), interrompendo o pipeline caso dados clinicamente impossíveis fossem detectados[cite: 4]. Além disso, realizou a remoção de duplicatas, tratamento de nulos e normalização dos nomes e valores das colunas[cite: 4].
* **Camada Gold (Business & ML Ready):** Estruturou *features* analíticas (como indicadores de hipertensão e faixas etárias) e codificou a variável alvo (`target_risk_encoded`)[cite: 4]. Aplicaram-se também rotinas de manutenção de performance Delta (`OPTIMIZE`)[cite: 4].

#### 2. Análise Exploratória e Diagnóstico de Dados
A análise de dados revelou que a idade materna isolada possui uma correlação positiva baixa (~0,18) com a classe de risco gestacional[cite: 4]. Essa evidência confirma a hipótese clínica de que o risco na gravidez é uma condição multifatorial — fortemente impulsionada pela combinação entre pressão arterial (sistólica/diastólica) e níveis de glicemia — o que justifica a necessidade de modelos preditivos não lineares[cite: 4].

#### 3. Ciência de Dados e Modelagem Preditiva
A comparação entre os algoritmos de aprendizado supervisionado (*Logistic Regression*, *Decision Tree* e *Random Forest*) destacou a complexidade de classificar perfis de risco com sintomas sobrepostos[cite: 4]. A aplicação de padronização (`StandardScaler`) e o balanceamento de pesos (`class_weight='balanced'`) foram determinantes para evitar que o modelo ficasse viesado a favor das classes mais comuns[cite: 4].

O modelo **Random Forest Otimizado** consagrou-se como o melhor classificador do projeto:
* **Acurácia Global:** 64,84%[cite: 4]
* **Sensibilidade (Recall) no Alto Risco:** ~83%[cite: 4]

---

### Interpretação Clínica e Impacto no Mundo Real

1. **O Valor do Recall Elevado no Alto Risco (~83%):**
   Na área médica, o **falso negativo** é a falha mais grave, pois significa enviar para casa uma gestante em situação crítica sem a devida assistência. 
   O fato de o *Random Forest* ter alcançado elevada sensibilidade garante que o modelo atue como uma **rede de proteção confiável**, identificando a grande maioria das pacientes de alto risco para triagem médica imediata.

2. **A Leitura da Acurácia de 64,84%:**
   Em dados de saúde, parâmetros fisiológicos de pacientes de *Médio Risco* se confundem constantemente com os de *Baixo* e *Alto Risco*. 
   A acurácia global de ~65% reflete uma escolha deliberada do modelo de **sacrificar a precisão perfeita no médio/baixo risco em prol de salvar vidas no alto risco**, maximizando a detecção das pacientes graves.

---

### Conclusão Geral
O projeto comprova a viabilidade técnica e prática do uso do **Databricks e Delta Lake** para construir sistemas de dados seguros, escaláveis e auditáveis.
 A solução entregue estabelece uma base sólida que une a engenharia robusta à inteligência de máquina, oferecendo um recurso capaz de apoiar equipes de saúde na tomada de decisões preventivas e na redução de complicações na saúde materna.